# Module-2 | Notebook 02 — Data Preprocessing

**Purpose:** Preprocess COCO 2014 and RefCOCO datasets into clean, structured DataFrames  
**Depends on:** `01_dataset_inspection.ipynb` (paths and dataset understanding)  
**Author:** NLP Research Project — Module 2  

---

### Ground Rules
- Raw files under `module-2/datasets/` are **READ-ONLY** — never modified, renamed, moved, or overwritten.
- All processed outputs are saved under `module-2/outputs/`.
- COCO train2014 is the **single source of truth** for images and annotations.
- RefCOCO data is linked to COCO via `image_id` and `ann_id` — no physical merging of images.

### Output Files
| File | Description |
|---|---|
| `outputs/coco_processed.csv` | Clean COCO image–annotation mapping |
| `outputs/refcoco_processed.csv` | Clean RefCOCO referring-expression records |

### Preprocessing Steps
1. Environment Setup
2. Load COCO 2014 Train Annotations
3. Build COCO Image–Annotation Mapping
4. Load RefCOCO Data
5. Build Structured RefCOCO DataFrame
6. Clean Referring Expressions
7. Validate Bounding Boxes
8. Validate Image and Annotation References
9. Handle Duplicates
10. Save Processed Data
11. Preprocessing Summary & Sanity Checks


---
## Step 1 — Environment Setup

**What:** Import all required libraries and establish consistent path constants.

**Why:** Centralising paths here ensures every downstream cell refers to the same locations, making the notebook reproducible across machines.

**Paths defined:**
- `DATASETS_DIR` — read-only root for all raw data
- `OUTPUTS_DIR` — write destination for all processed artefacts
- `COCO_*` — paths to COCO annotation files
- `REFCOCO_*` — paths to RefCOCO files

**Expected output:** All path existence checks print `True`.


In [26]:
# Standard Library
import sys
import os
import json
import pickle
import re
import warnings
from pathlib import Path
from collections import Counter, defaultdict

# Third-party
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f'Python  : {sys.version}')
print(f'NumPy   : {np.__version__}')
print(f'Pandas  : {pd.__version__}')


Python  : 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
NumPy   : 2.3.2
Pandas  : 2.3.1


In [27]:
# ── Path Definitions (mirrors 01_dataset_inspection.ipynb) ──────────────────
NOTEBOOK_DIR  = Path(os.getcwd())           # module-2/ipynb/
MODULE2_DIR   = NOTEBOOK_DIR.parent         # module-2/
DATASETS_DIR  = MODULE2_DIR / 'datasets'    # READ-ONLY
OUTPUTS_DIR   = MODULE2_DIR / 'outputs'     # all processed outputs go here

# COCO 2014
COCO_DIR        = DATASETS_DIR / 'coco2014'
COCO_TRAIN_DIR  = COCO_DIR / 'train2014'
COCO_ANN_DIR    = COCO_DIR / 'annotations'
COCO_INST_TRAIN = COCO_ANN_DIR / 'instances_train2014.json'

# RefCOCO
REFCOCO_DIR    = DATASETS_DIR / 'refcoco' / 'refcoco'
REFCOCO_INST   = REFCOCO_DIR / 'instances.json'
REFCOCO_UNC    = REFCOCO_DIR / 'refs(unc).p'
REFCOCO_GOOGLE = REFCOCO_DIR / 'refs(google).p'

# Output paths for this notebook
COCO_PROCESSED_CSV    = OUTPUTS_DIR / 'coco_processed.csv'
REFCOCO_PROCESSED_CSV = OUTPUTS_DIR / 'refcoco_processed.csv'

# Ensure outputs directory exists
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print('PATH VALIDATION')
print('=' * 60)
checks = [
    ('DATASETS_DIR  (read-only)', DATASETS_DIR),
    ('OUTPUTS_DIR   (writable) ', OUTPUTS_DIR),
    ('COCO train2014/          ', COCO_TRAIN_DIR),
    ('COCO annotations/        ', COCO_ANN_DIR),
    ('instances_train2014.json ', COCO_INST_TRAIN),
    ('RefCOCO instances.json   ', REFCOCO_INST),
    ('refs(unc).p              ', REFCOCO_UNC),
    ('refs(google).p           ', REFCOCO_GOOGLE),
]
all_ok = True
for label, path in checks:
    ok = path.exists()
    all_ok = all_ok and ok
    status = 'OK' if ok else 'MISSING'
    print(f'  [{status}] {label} : {path}')
print()
print('All paths valid:', all_ok)


PATH VALIDATION
  [OK] DATASETS_DIR  (read-only) : d:\NLP-Research\module-2\datasets
  [OK] OUTPUTS_DIR   (writable)  : d:\NLP-Research\module-2\outputs
  [OK] COCO train2014/           : d:\NLP-Research\module-2\datasets\coco2014\train2014
  [OK] COCO annotations/         : d:\NLP-Research\module-2\datasets\coco2014\annotations
  [OK] instances_train2014.json  : d:\NLP-Research\module-2\datasets\coco2014\annotations\instances_train2014.json
  [OK] RefCOCO instances.json    : d:\NLP-Research\module-2\datasets\refcoco\refcoco\instances.json
  [OK] refs(unc).p               : d:\NLP-Research\module-2\datasets\refcoco\refcoco\refs(unc).p
  [OK] refs(google).p            : d:\NLP-Research\module-2\datasets\refcoco\refcoco\refs(google).p

All paths valid: True


---
## Step 2 — Load COCO 2014 Train Annotations

**What:** Load `instances_train2014.json` and extract the three lists it contains.

**Why:** These lists are the source of truth for all image metadata, object annotations,
and category definitions. Every downstream step — including RefCOCO linking — depends on them.

**Variables after this step:**
- `coco_images` — list of dicts, one per image (keys: `id`, `file_name`, `width`, `height`, ...)
- `coco_annotations` — list of dicts, one per object instance (keys: `id`, `image_id`, `category_id`, `bbox`, `area`, ...)
- `coco_categories` — list of dicts, one per category (keys: `id`, `name`, `supercategory`)

**Expected output:** Counts matching what was found in notebook 01.


In [28]:
print('Loading instances_train2014.json  (may take ~15-20 s) ...')
with open(COCO_INST_TRAIN, 'r') as f:
    coco_data = json.load(f)
print('Loaded.')

coco_images      = coco_data['images']
coco_annotations = coco_data['annotations']
coco_categories  = coco_data['categories']

print()
print('COCO 2014 — Load Summary')
print('=' * 40)
print(f'  Images      : {len(coco_images):>8,}')
print(f'  Annotations : {len(coco_annotations):>8,}')
print(f'  Categories  : {len(coco_categories):>8,}')


Loading instances_train2014.json  (may take ~15-20 s) ...
Loaded.

COCO 2014 — Load Summary
  Images      :   82,783
  Annotations :  604,907
  Categories  :       80


### 2.1 — Build Efficient Lookup Dictionaries

**What:** Convert the flat lists into dictionaries keyed by `id` for O(1) lookups.

**Why:** The lists contain tens of thousands of records. Dict lookups are orders of magnitude
faster than list scans — critical when we later cross-reference every RefCOCO entry.

**Variables:**
- `img_lookup` — `{image_id: image_dict}` — fast retrieval of any COCO image by its ID
- `ann_lookup` — `{ann_id: annotation_dict}` — fast retrieval of any annotation by its ID
- `cat_lookup` — `{category_id: category_name}` — fast category ID → name resolution
- `img_to_anns` — `{image_id: [ann, ann, ...]}` — all annotations for a given image


In [29]:
# image_id -> image dict
img_lookup  = {img['id']: img for img in coco_images}

# ann_id -> annotation dict
ann_lookup  = {ann['id']: ann for ann in coco_annotations}

# category_id -> category name
cat_lookup  = {cat['id']: cat['name'] for cat in coco_categories}

# image_id -> list of annotation dicts
img_to_anns = defaultdict(list)
for ann in coco_annotations:
    img_to_anns[ann['image_id']].append(ann)

print(f'img_lookup  : {len(img_lookup):,} entries')
print(f'ann_lookup  : {len(ann_lookup):,} entries')
print(f'cat_lookup  : {len(cat_lookup):,} entries')
print(f'img_to_anns : {len(img_to_anns):,} keys  '
      f'(avg {sum(len(v) for v in img_to_anns.values()) / len(img_to_anns):.1f} anns/image)')


img_lookup  : 82,783 entries
ann_lookup  : 604,907 entries
cat_lookup  : 80 entries
img_to_anns : 82,081 keys  (avg 7.4 anns/image)


---
## Step 3 — Build COCO Image–Annotation Mapping

**What:** Flatten every COCO image + annotation pair into a single row with all useful fields.

**Why:** A flat tabular representation makes downstream filtering, validation,
and saving to CSV straightforward. It also serves as the canonical reference
for image dimensions and bounding box values used in RefCOCO validation.

**Columns produced:**
| Column | Source | Description |
|---|---|---|
| `image_id` | `images[].id` | COCO image identifier |
| `image_path` | `images[].file_name` | Filename relative to `train2014/` |
| `width` | `images[].width` | Image width in pixels |
| `height` | `images[].height` | Image height in pixels |
| `ann_id` | `annotations[].id` | Annotation (object instance) identifier |
| `bbox_x` | `annotations[].bbox[0]` | Bounding box left edge |
| `bbox_y` | `annotations[].bbox[1]` | Bounding box top edge |
| `bbox_width` | `annotations[].bbox[2]` | Bounding box width |
| `bbox_height` | `annotations[].bbox[3]` | Bounding box height |
| `category_id` | `annotations[].category_id` | Numeric category identifier |
| `category_name` | `categories[].name` | Human-readable category label |

**Expected output:** One row per annotation (~604 k rows for train2014).


In [30]:
print('Building COCO image-annotation mapping ...')

coco_rows = []
for ann in coco_annotations:
    img = img_lookup.get(ann['image_id'])
    if img is None:
        continue  # orphan annotation — extremely rare but safe to skip
    bbox = ann['bbox']  # [x, y, w, h]
    coco_rows.append({
        'image_id'     : ann['image_id'],
        'image_path'   : img['file_name'],
        'width'        : img['width'],
        'height'       : img['height'],
        'ann_id'       : ann['id'],
        'bbox_x'       : bbox[0],
        'bbox_y'       : bbox[1],
        'bbox_width'   : bbox[2],
        'bbox_height'  : bbox[3],
        'category_id'  : ann['category_id'],
        'category_name': cat_lookup.get(ann['category_id'], 'unknown'),
    })

coco_df_raw = pd.DataFrame(coco_rows)

print(f'Rows in COCO mapping : {len(coco_df_raw):,}')
print(f'Columns              : {list(coco_df_raw.columns)}')
print()
print(coco_df_raw.head(5).to_string(index=False))


Building COCO image-annotation mapping ...
Rows in COCO mapping : 604,907
Columns              : ['image_id', 'image_path', 'width', 'height', 'ann_id', 'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height', 'category_id', 'category_name']

 image_id                      image_path  width  height  ann_id   bbox_x   bbox_y  bbox_width  bbox_height  category_id category_name
   480023 COCO_train2014_000000480023.jpg    480     640      86 116.9500 305.8600    285.3000     266.0300           58       hot dog
    50518 COCO_train2014_000000050518.jpg    640     480      89 245.5400 208.1700     40.1400      19.1000           58       hot dog
   142589 COCO_train2014_000000142589.jpg    500     375      93 288.4000  18.0700    211.6000     331.3300           58       hot dog
   209263 COCO_train2014_000000209263.jpg    480     640     113 126.5000 475.2400     77.6800      76.7300           58       hot dog
    15307 COCO_train2014_000000015307.jpg    612     612     116 185.5700  93.4000    219.

In [31]:
print('COCO DataFrame — dtype summary')
print(coco_df_raw.dtypes)
print()
print('Null counts:')
print(coco_df_raw.isnull().sum())
print()
print('Unique counts:')
print(f'  Unique image_ids    : {coco_df_raw["image_id"].nunique():,}')
print(f'  Unique ann_ids      : {coco_df_raw["ann_id"].nunique():,}')
print(f'  Unique categories   : {coco_df_raw["category_name"].nunique():,}')


COCO DataFrame — dtype summary
image_id           int64
image_path        object
width              int64
height             int64
ann_id             int64
bbox_x           float64
bbox_y           float64
bbox_width       float64
bbox_height      float64
category_id        int64
category_name     object
dtype: object

Null counts:
image_id         0
image_path       0
width            0
height           0
ann_id           0
bbox_x           0
bbox_y           0
bbox_width       0
bbox_height      0
category_id      0
category_name    0
dtype: int64

Unique counts:
  Unique image_ids    : 82,081
  Unique ann_ids      : 604,907
  Unique categories   : 80


---
## Step 4 — Load RefCOCO Data

**What:** Load the three RefCOCO files — `instances.json`, `refs(unc).p` — and
flatten each reference into a row.

**Why:** RefCOCO provides referring expressions for specific COCO objects.
Loading both `instances.json` (for any RefCOCO-specific annotation metadata) and
`refs(unc).p` (for expressions and splits) gives us a complete picture.

**Key RefCOCO ref fields (from inspection notebook):**
- `image_id` — COCO image ID the expression refers to
- `ann_id` — COCO annotation (object instance) ID being referred to
- `sentences` — list of expression dicts; each has a `sent` field
- `split` — dataset split (`train`, `val`, `testA`, `testB`)

**Design decision — source column:**  
We load both `refs(unc).p` and `refs(google).p`. Each row will be tagged with
`source='unc'` or `source='google'` so they can be filtered independently downstream.

**Expected output:** Two lists of ref-entry dicts ready for DataFrame construction.


In [32]:
def load_pickle_refs(path, encoding='latin1'):
    """Load a RefCOCO .p file and normalise it to a list of ref dicts."""
    with open(path, 'rb') as fh:
        data = pickle.load(fh, encoding=encoding)
    return data if isinstance(data, list) else list(data.values())


print('Loading refs(unc).p    ...')
refs_unc    = load_pickle_refs(REFCOCO_UNC)
print(f'  Loaded {len(refs_unc):,} UNC refs')

print('Loading refs(google).p ...')
refs_google = load_pickle_refs(REFCOCO_GOOGLE)
print(f'  Loaded {len(refs_google):,} Google refs')

print()
print('Loading RefCOCO instances.json ...')
with open(REFCOCO_INST, 'r') as fh:
    refcoco_inst = json.load(fh)
print(f'  images      : {len(refcoco_inst.get("images", [])):,}')
print(f'  annotations : {len(refcoco_inst.get("annotations", [])):,}')
print(f'  categories  : {len(refcoco_inst.get("categories", [])):,}')


Loading refs(unc).p    ...
  Loaded 50,000 UNC refs
Loading refs(google).p ...
  Loaded 50,000 Google refs

Loading RefCOCO instances.json ...
  images      : 19,994
  annotations : 196,771
  categories  : 80


### 4.1 — Extract Referring Expressions from Each Source

**What:** Convert each ref entry's `sentences` list into individual rows, one per expression.

**Why:** A single annotation (object) can have multiple referring expressions written by
different annotators. Expanding them into separate rows gives one expression per row — the
natural unit for an NLP model.

**Variable `raw_refcoco_rows`:** List of dicts with one entry per (ref, expression) pair.
Each dict contains `image_id`, `ann_id`, `split`, `source`, and `referring_expression`.


In [33]:
def extract_rows_from_refs(refs_list, source_label):
    """
    Flatten a list of RefCOCO ref dicts into one row per referring expression.

    Parameters
    ----------
    refs_list    : list of ref dicts (from pickle)
    source_label : str — 'unc' or 'google'

    Returns
    -------
    list of dicts with keys: image_id, ann_id, split, source, referring_expression
    """
    rows = []
    for ref in refs_list:
        image_id = ref.get('image_id')
        ann_id   = ref.get('ann_id')
        split    = ref.get('split', 'unknown')
        sentences = ref.get('sentences', [])

        for sent in sentences:
            # 'sent' is the clean sentence; 'raw' is the pre-tokenisation form
            expression = sent.get('sent', sent.get('raw', None))
            rows.append({
                'image_id'            : image_id,
                'ann_id'              : ann_id,
                'split'               : split,
                'source'              : source_label,
                'referring_expression': expression,
            })
    return rows


rows_unc    = extract_rows_from_refs(refs_unc,    source_label='unc')
rows_google = extract_rows_from_refs(refs_google, source_label='google')

print(f'UNC rows    (expressions) : {len(rows_unc):,}')
print(f'Google rows (expressions) : {len(rows_google):,}')

raw_refcoco_rows = rows_unc + rows_google
print(f'Total raw rows combined   : {len(raw_refcoco_rows):,}')

# Record original counts for the summary table at the end
ORIG_COCO_IMAGES      = len(coco_images)
ORIG_COCO_ANNS        = len(coco_annotations)
ORIG_REFCOCO_REFS_UNC = len(refs_unc)
ORIG_REFCOCO_REFS_GGL = len(refs_google)
ORIG_REFCOCO_EXPRS    = len(raw_refcoco_rows)


UNC rows    (expressions) : 142,210
Google rows (expressions) : 142,210
Total raw rows combined   : 284,420


---
## Step 5 — Build Structured RefCOCO DataFrame

**What:** Join each (image_id, ann_id) pair from RefCOCO against the COCO lookups to enrich
every expression row with image metadata and bounding box information.

**Why:** The referring expression alone is not sufficient for model training — we need:
- The image path (to load the visual input)
- The bounding box (as the ground-truth target region)
- The category (for evaluation and stratification)

All of these come from COCO. We do **not** copy image pixels — only metadata.

**Columns in `refcoco_df_raw`:**
| Column | Description |
|---|---|
| `image_id` | COCO image identifier |
| `image_path` | Filename under `train2014/` |
| `ann_id` | COCO annotation (object) identifier |
| `bbox_x/y/width/height` | Bounding box in COCO format |
| `category_id/name` | Object class |
| `referring_expression` | Raw expression string from annotators |
| `split` | Dataset split (`train`/`val`/`testA`/`testB`) |
| `source` | Annotation source (`unc` / `google`) |

**Expected output:** One row per (expression, annotation) pair.


In [34]:
print('Building RefCOCO DataFrame — joining with COCO lookups ...')

refcoco_rows_enriched = []
n_missing_img = 0
n_missing_ann = 0

for row in raw_refcoco_rows:
    img_id = row['image_id']
    ann_id = row['ann_id']

    coco_img = img_lookup.get(img_id)
    coco_ann = ann_lookup.get(ann_id)

    if coco_img is None:
        n_missing_img += 1
        continue
    if coco_ann is None:
        n_missing_ann += 1
        continue

    bbox = coco_ann['bbox']  # [x, y, w, h]
    refcoco_rows_enriched.append({
        'image_id'            : img_id,
        'image_path'          : coco_img['file_name'],
        'width'               : coco_img['width'],
        'height'              : coco_img['height'],
        'ann_id'              : ann_id,
        'bbox_x'              : bbox[0],
        'bbox_y'              : bbox[1],
        'bbox_width'          : bbox[2],
        'bbox_height'         : bbox[3],
        'category_id'         : coco_ann['category_id'],
        'category_name'       : cat_lookup.get(coco_ann['category_id'], 'unknown'),
        'referring_expression': row['referring_expression'],
        'split'               : row['split'],
        'source'              : row['source'],
    })

refcoco_df_raw = pd.DataFrame(refcoco_rows_enriched)

print(f'Rows built          : {len(refcoco_df_raw):,}')
print(f'Dropped (no image)  : {n_missing_img:,}')
print(f'Dropped (no ann)    : {n_missing_ann:,}')
print()
print('Columns:', list(refcoco_df_raw.columns))
print()
print(refcoco_df_raw.head(5).to_string(index=False))


Building RefCOCO DataFrame — joining with COCO lookups ...
Rows built          : 284,420
Dropped (no image)  : 0
Dropped (no ann)    : 0

Columns: ['image_id', 'image_path', 'width', 'height', 'ann_id', 'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height', 'category_id', 'category_name', 'referring_expression', 'split', 'source']

 image_id                      image_path  width  height  ann_id   bbox_x   bbox_y  bbox_width  bbox_height  category_id category_name                       referring_expression split source
   581857 COCO_train2014_000000581857.jpg    427     640 1719310 103.9300 299.9900    134.2200     177.4200            1        person               the lady with the blue shirt train    unc
   581857 COCO_train2014_000000581857.jpg    427     640 1719310 103.9300 299.9900    134.2200     177.4200            1        person                       lady with back to us train    unc
   581857 COCO_train2014_000000581857.jpg    427     640 1719310 103.9300 299.9900    134.2200     

In [35]:
print('RefCOCO DataFrame — dtype summary')
print(refcoco_df_raw.dtypes)
print()
print('Null counts before cleaning:')
print(refcoco_df_raw.isnull().sum())
print()
print('Split x Source distribution:')
print(refcoco_df_raw.groupby(['split', 'source']).size().to_string())


RefCOCO DataFrame — dtype summary
image_id                  int64
image_path               object
width                     int64
height                    int64
ann_id                    int64
bbox_x                  float64
bbox_y                  float64
bbox_width              float64
bbox_height             float64
category_id               int64
category_name            object
referring_expression     object
split                    object
source                   object
dtype: object

Null counts before cleaning:
image_id                0
image_path              0
width                   0
height                  0
ann_id                  0
bbox_x                  0
bbox_y                  0
bbox_width              0
bbox_height             0
category_id             0
category_name           0
referring_expression    0
split                   0
source                  0
dtype: int64

Split x Source distribution:
split  source
test   google     14202
testA  unc         5657
testB

---
## Step 6 — Clean Referring Expressions

**What:** Apply light, semantics-preserving normalisation to the `referring_expression` column.

**Why:** Raw crowdsourced expressions may contain:
- Leading / trailing whitespace
- Collapsed multiple spaces (e.g. `'the  big  dog'`)
- Null / empty values from failed sentence extraction

**What we deliberately do NOT do:**
- Lowercasing — preserves proper nouns and acronyms
- Punctuation removal — `'left,'` and `'left'` can mean different things to a model
- Stopword removal — referential language (`'the person on the left'`) depends on stopwords
- Stemming / lemmatisation — premature for this stage

**Variable `refcoco_df`:** Working copy after expression cleaning.

**Expected output:** Null count for `referring_expression` drops to 0; whitespace normalised.


In [36]:
refcoco_df = refcoco_df_raw.copy()

print('Before cleaning:')
print(f'  Total rows              : {len(refcoco_df):,}')
print(f'  Null expressions        : {refcoco_df["referring_expression"].isnull().sum():,}')
print(f'  Empty string expressions: '
      f'{(refcoco_df["referring_expression"] == "").sum():,}')

# ── 1. Strip leading/trailing whitespace ──────────────────────────────────────
refcoco_df['referring_expression'] = (
    refcoco_df['referring_expression']
    .astype(str)  # cast None/NaN to the string 'None'/'nan' first
    .str.strip()
)

# ── 2. Normalise internal whitespace (collapse multiple spaces / tabs) ─────────
refcoco_df['referring_expression'] = refcoco_df['referring_expression'].apply(
    lambda x: re.sub(r'\s+', ' ', x)
)

# ── 3. Mark sentinel strings from the astype(str) cast as empty ───────────────
refcoco_df['referring_expression'] = refcoco_df['referring_expression'].replace(
    {'None': '', 'nan': '', 'NaN': ''}
)

# ── 4. Record and remove null / empty expressions ─────────────────────────────
mask_empty = refcoco_df['referring_expression'] == ''
n_empty    = mask_empty.sum()

print()
print('After whitespace normalisation:')
print(f'  Empty expressions to remove : {n_empty:,}')

if n_empty > 0:
    print('  Sample empty rows:')
    print(refcoco_df[mask_empty].head(3).to_string(index=False))

refcoco_df = refcoco_df[~mask_empty].copy()
refcoco_df.reset_index(drop=True, inplace=True)

print()
print(f'Rows remaining after expression cleaning : {len(refcoco_df):,}')


Before cleaning:
  Total rows              : 284,420
  Null expressions        : 0
  Empty string expressions: 0

After whitespace normalisation:
  Empty expressions to remove : 0

Rows remaining after expression cleaning : 284,420


In [37]:
# Sanity-check: sample cleaned expressions
print('Sample cleaned referring expressions:')
sample_exprs = refcoco_df[['image_id', 'ann_id', 'split', 'source', 'referring_expression']]
print(sample_exprs.sample(10, random_state=42).to_string(index=False))


Sample cleaned referring expressions:
 image_id  ann_id split source               referring_expression
   487244 1214618  test google            the area above the hand
   179159  581646 train google          one on left in foreground
    95611   59412 train google               horse looking at cam
    76882  421633  test google                           left guy
    40906  447679 train    unc man in blue and white floral shirt
   275556  466439   val google          blond on left wcurly hair
   117349  512783 train google             woman front right side
     6225  454314 train    unc          guy on right in gray vest
   493601 2219162 train    unc                     top left bread
   478968  512918 train    unc                  woman in the back


---
## Step 7 — Validate Bounding Boxes

**What:** Check every bounding box against two criteria:
1. **Positive dimensions** — `bbox_width > 0` and `bbox_height > 0`
2. **Within-image bounds** — the box must not extend beyond the image edges

**Why:** A zero-area or out-of-bounds box cannot represent a valid region of interest and
would cause errors or undefined behaviour during model training. We report all violations
before removing them so nothing is discarded silently.

**COCO bbox format:** `[x_min, y_min, width, height]`  
→ right edge  = `x_min + width`  
→ bottom edge = `y_min + height`

**Tolerance:** We allow `1 px` of overshoot to absorb floating-point rounding from COCO annotations.

**Expected output:** Invalid bbox count reported; valid-only DataFrame retained.


In [38]:
def validate_bboxes(df, tolerance=1.0):
    """
    Validate bounding boxes in a DataFrame.

    Rules
    -----
    - bbox_width  > 0
    - bbox_height > 0
    - bbox_x >= 0
    - bbox_y >= 0
    - bbox_x + bbox_width  <= width  + tolerance
    - bbox_y + bbox_height <= height + tolerance

    Returns
    -------
    mask_valid : boolean Series — True where bbox is valid
    report     : dict with counts of each failure type
    """
    report = {}

    bad_w    = df['bbox_width']  <= 0
    bad_h    = df['bbox_height'] <= 0
    bad_x    = df['bbox_x'] < 0
    bad_y    = df['bbox_y'] < 0
    overflow_x = (df['bbox_x'] + df['bbox_width'])  > (df['width']  + tolerance)
    overflow_y = (df['bbox_y'] + df['bbox_height']) > (df['height'] + tolerance)

    report['zero_width']    = int(bad_w.sum())
    report['zero_height']   = int(bad_h.sum())
    report['negative_x']   = int(bad_x.sum())
    report['negative_y']   = int(bad_y.sum())
    report['overflow_x']   = int(overflow_x.sum())
    report['overflow_y']   = int(overflow_y.sum())

    mask_invalid = bad_w | bad_h | bad_x | bad_y | overflow_x | overflow_y
    report['total_invalid'] = int(mask_invalid.sum())

    return ~mask_invalid, report


# ── Validate RefCOCO bboxes ───────────────────────────────────────────────────
valid_mask_rc, bbox_report_rc = validate_bboxes(refcoco_df)

print('BOUNDING BOX VALIDATION — RefCOCO')
print('=' * 50)
for reason, count in bbox_report_rc.items():
    print(f'  {reason:<20} : {count:,}')

if bbox_report_rc['total_invalid'] > 0:
    print()
    print('Sample invalid rows:')
    bad_rows = refcoco_df[~valid_mask_rc][
        ['image_id', 'ann_id', 'bbox_x', 'bbox_y',
         'bbox_width', 'bbox_height', 'width', 'height']
    ].head(5)
    print(bad_rows.to_string(index=False))

n_bbox_removed_rc = bbox_report_rc['total_invalid']
refcoco_df = refcoco_df[valid_mask_rc].copy()
refcoco_df.reset_index(drop=True, inplace=True)

print()
print(f'Rows remaining after bbox validation : {len(refcoco_df):,}')


BOUNDING BOX VALIDATION — RefCOCO
  zero_width           : 0
  zero_height          : 0
  negative_x           : 0
  negative_y           : 0
  overflow_x           : 0
  overflow_y           : 0
  total_invalid        : 0

Rows remaining after bbox validation : 284,420


In [39]:
# ── Validate COCO bboxes as well ─────────────────────────────────────────────
valid_mask_coco, bbox_report_coco = validate_bboxes(coco_df_raw)

print('BOUNDING BOX VALIDATION — COCO')
print('=' * 50)
for reason, count in bbox_report_coco.items():
    print(f'  {reason:<20} : {count:,}')

if bbox_report_coco['total_invalid'] > 0:
    print()
    print('Sample invalid COCO bbox rows:')
    print(coco_df_raw[~valid_mask_coco].head(5).to_string(index=False))

n_bbox_removed_coco = bbox_report_coco['total_invalid']
coco_df = coco_df_raw[valid_mask_coco].copy()
coco_df.reset_index(drop=True, inplace=True)

print()
print(f'Rows remaining after bbox validation : {len(coco_df):,}')


BOUNDING BOX VALIDATION — COCO
  zero_width           : 0
  zero_height          : 1
  negative_x           : 0
  negative_y           : 0
  overflow_x           : 0
  overflow_y           : 0
  total_invalid        : 1

Sample invalid COCO bbox rows:
 image_id                      image_path  width  height  ann_id  bbox_x   bbox_y  bbox_width  bbox_height  category_id category_name
   550395 COCO_train2014_000000550395.jpg    640     429 2206849  9.9800 188.5600      5.5400       0.0000            1        person

Rows remaining after bbox validation : 604,906


---
## Step 8 — Validate Image and Annotation References

**What:** Three referential integrity checks on the RefCOCO DataFrame:

1. Every `image_id` in RefCOCO must exist in the COCO `img_lookup`.
2. Every `ann_id` in RefCOCO must exist in the COCO `ann_lookup`.
3. The `image_id` embedded in the COCO annotation must match the RefCOCO `image_id`
   (checks that the ann really belongs to the referenced image).

**Why:** Any mismatch would mean the expression is pointing at an object in the wrong image,
or pointing at an object that does not exist in our copy of COCO — both are silent data errors
that would corrupt model training.

**Expected output:** All checks should pass at ≥99% rate based on inspection notebook findings.
Any failures are reported and removed with an explicit count.


In [40]:
print('REFERENTIAL INTEGRITY VALIDATION — RefCOCO')
print('=' * 55)

total_before = len(refcoco_df)

# Check 1: image_id exists in COCO
mask_img_ok = refcoco_df['image_id'].isin(img_lookup)
n_bad_img   = (~mask_img_ok).sum()
print(f'  Check 1 — image_id in COCO  : {n_bad_img:,} mismatches')
if n_bad_img > 0:
    bad_ids = refcoco_df.loc[~mask_img_ok, 'image_id'].unique()[:5]
    print(f'  Sample missing image_ids: {bad_ids}')

# Check 2: ann_id exists in COCO
mask_ann_ok = refcoco_df['ann_id'].isin(ann_lookup)
n_bad_ann   = (~mask_ann_ok).sum()
print(f'  Check 2 — ann_id in COCO    : {n_bad_ann:,} mismatches')
if n_bad_ann > 0:
    bad_ids = refcoco_df.loc[~mask_ann_ok, 'ann_id'].unique()[:5]
    print(f'  Sample missing ann_ids: {bad_ids}')

# Check 3: annotation image_id matches RefCOCO image_id
# Use vectorised lookup via a helper column
refcoco_df['_ann_image_id'] = refcoco_df['ann_id'].map(
    lambda aid: ann_lookup[aid]['image_id'] if aid in ann_lookup else -1
)
mask_match_ok = refcoco_df['_ann_image_id'] == refcoco_df['image_id']
n_bad_match   = (~mask_match_ok).sum()
print(f'  Check 3 — ann.image_id match : {n_bad_match:,} mismatches')
if n_bad_match > 0:
    print('  Sample mismatched rows:')
    print(refcoco_df.loc[~mask_match_ok,
                         ['image_id', 'ann_id', '_ann_image_id']].head(5).to_string(index=False))

# Remove the helper column
refcoco_df.drop(columns=['_ann_image_id'], inplace=True)

# Combine all validity masks
mask_all_valid  = mask_img_ok & mask_ann_ok & mask_match_ok
n_ref_removed   = int((~mask_all_valid).sum())

refcoco_df = refcoco_df[mask_all_valid].copy()
refcoco_df.reset_index(drop=True, inplace=True)

print()
print(f'Rows before reference validation : {total_before:,}')
print(f'Rows removed (ref integrity)     : {n_ref_removed:,}')
print(f'Rows remaining                   : {len(refcoco_df):,}')


REFERENTIAL INTEGRITY VALIDATION — RefCOCO
  Check 1 — image_id in COCO  : 0 mismatches


  Check 2 — ann_id in COCO    : 0 mismatches
  Check 3 — ann.image_id match : 0 mismatches

Rows before reference validation : 284,420
Rows removed (ref integrity)     : 0
Rows remaining                   : 284,420


---
## Step 9 — Handle Duplicates

**What:** Identify and remove **exact duplicate rows** — rows where every column is identical.

**Why:**  
- Exact duplicates provide no additional information and inflate epoch counts during training.
- However, it is expected and intentional that the same object (`ann_id`) has **multiple different
  referring expressions** — those are NOT duplicates and must be preserved.
- We define a duplicate as a row where `(ann_id, referring_expression, source)` are all identical,
  meaning the same annotator wrote the same sentence for the same object twice.

**Deduplication key:** `['ann_id', 'referring_expression', 'source']`  
Rationale: Different sources (unc vs google) writing the same expression for the same object
are kept because they provide independent annotation evidence. Only within-source duplicates
are removed.

**Expected output:** Small number of exact duplicates (if any) removed and reported.


In [41]:
# ── Analyse duplicates ───────────────────────────────────────────────────────
DUP_KEY = ['ann_id', 'referring_expression', 'source']

n_total        = len(refcoco_df)
n_exact_dups   = refcoco_df.duplicated(subset=DUP_KEY, keep='first').sum()

print('DUPLICATE ANALYSIS — RefCOCO')
print('=' * 55)
print(f'  Total rows             : {n_total:,}')
print(f'  Exact duplicates found : {n_exact_dups:,}')
print(f'    (key: {DUP_KEY})')

if n_exact_dups > 0:
    print()
    print('  Sample duplicate pairs:')
    dup_mask = refcoco_df.duplicated(subset=DUP_KEY, keep=False)
    dup_sample = (
        refcoco_df[dup_mask]
        .sort_values(DUP_KEY)
        .head(10)
    )
    print(dup_sample[DUP_KEY + ['split', 'image_id']].to_string(index=False))

# Remove duplicates (keep first occurrence)
refcoco_df = refcoco_df.drop_duplicates(subset=DUP_KEY, keep='first').copy()
refcoco_df.reset_index(drop=True, inplace=True)

print()
print(f'Rows remaining after deduplication : {len(refcoco_df):,}')


DUPLICATE ANALYSIS — RefCOCO
  Total rows             : 284,420
  Exact duplicates found : 16,852
    (key: ['ann_id', 'referring_expression', 'source'])

  Sample duplicate pairs:
 ann_id referring_expression source split  image_id
    765        left sandwich google  test     10046
    765        left sandwich google  test     10046
    765        left sandwich    unc train     10046
    765        left sandwich    unc train     10046
   1270        right hot dog google train     46298
   1270        right hot dog google train     46298
   1270        right hot dog    unc train     46298
   1270        right hot dog    unc train     46298
   1610          dog on left google  test    288127
   1610          dog on left google  test    288127

Rows remaining after deduplication : 267,568


In [42]:
# ── Duplicates in COCO DataFrame ─────────────────────────────────────────────
COCO_DUP_KEY = ['ann_id']  # ann_id is the primary key — should be unique

n_coco_dups = coco_df.duplicated(subset=COCO_DUP_KEY, keep='first').sum()
print(f'COCO DataFrame — duplicate ann_ids : {n_coco_dups:,}')

if n_coco_dups > 0:
    coco_df = coco_df.drop_duplicates(subset=COCO_DUP_KEY, keep='first').copy()
    coco_df.reset_index(drop=True, inplace=True)
    print(f'  Removed {n_coco_dups:,} duplicate COCO rows.')
else:
    print('  No duplicates found — COCO ann_id is unique as expected.')


COCO DataFrame — duplicate ann_ids : 0
  No duplicates found — COCO ann_id is unique as expected.


---
## Step 10 — Save Processed Data

**What:** Write the two cleaned DataFrames to CSV files in `module-2/outputs/`.

**Why:** Persisting the processed data:
- Allows subsequent notebooks (EDA, model training) to read the clean data directly
  without re-running the expensive JSON loading and validation steps.
- Provides a transparent checkpoint — anyone can open the CSV and verify the data.

**Output files:**
- `outputs/coco_processed.csv` — COCO image–annotation mapping (one row per annotation)
- `outputs/refcoco_processed.csv` — RefCOCO expression records (one row per expression)

**Expected output:** File sizes in MB reported; no exceptions raised.


In [43]:
# ── Final column ordering ─────────────────────────────────────────────────────
COCO_COLS = [
    'image_id', 'image_path', 'width', 'height',
    'ann_id', 'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height',
    'category_id', 'category_name',
]

REFCOCO_COLS = [
    'image_id', 'image_path',
    'ann_id', 'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height',
    'category_id', 'category_name',
    'referring_expression', 'split', 'source',
]

coco_df    = coco_df[COCO_COLS]
refcoco_df = refcoco_df[REFCOCO_COLS]

print('Final column order — COCO    :', list(coco_df.columns))
print('Final column order — RefCOCO :', list(refcoco_df.columns))


Final column order — COCO    : ['image_id', 'image_path', 'width', 'height', 'ann_id', 'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height', 'category_id', 'category_name']
Final column order — RefCOCO : ['image_id', 'image_path', 'ann_id', 'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height', 'category_id', 'category_name', 'referring_expression', 'split', 'source']


In [44]:
# ── Save COCO processed ───────────────────────────────────────────────────────
print(f'Saving COCO processed -> {COCO_PROCESSED_CSV}')
coco_df.to_csv(COCO_PROCESSED_CSV, index=False)
coco_size_mb = COCO_PROCESSED_CSV.stat().st_size / (1024 ** 2)
print(f'  Rows    : {len(coco_df):,}')
print(f'  Columns : {len(coco_df.columns)}')
print(f'  Size    : {coco_size_mb:.2f} MB')

print()

# ── Save RefCOCO processed ────────────────────────────────────────────────────
print(f'Saving RefCOCO processed -> {REFCOCO_PROCESSED_CSV}')
refcoco_df.to_csv(REFCOCO_PROCESSED_CSV, index=False)
refcoco_size_mb = REFCOCO_PROCESSED_CSV.stat().st_size / (1024 ** 2)
print(f'  Rows    : {len(refcoco_df):,}')
print(f'  Columns : {len(refcoco_df.columns)}')
print(f'  Size    : {refcoco_size_mb:.2f} MB')


Saving COCO processed -> d:\NLP-Research\module-2\outputs\coco_processed.csv
  Rows    : 604,906
  Columns : 11
  Size    : 52.17 MB

Saving RefCOCO processed -> d:\NLP-Research\module-2\outputs\refcoco_processed.csv
  Rows    : 267,568
  Columns : 12
  Size    : 28.92 MB


---
## Step 11 — Preprocessing Summary & Sanity Checks

**What:** Consolidate all counts from this notebook into a single summary table
and perform final sanity checks.

**Why:** A clear before/after table makes it easy to audit what was removed and why,
and confirms the pipeline ran correctly end-to-end.

**Sanity checks:**
- Output CSV files exist and are non-empty
- Sample rows from each CSV load correctly
- No null values in critical columns of the final DataFrames
- Split distribution in RefCOCO looks reasonable


In [45]:
# ── Compute summary numbers ───────────────────────────────────────────────────
summary_rows = [
    # COCO
    ('COCO 2014', 'Original images',                  ORIG_COCO_IMAGES,  ''),
    ('COCO 2014', 'Original annotations',              ORIG_COCO_ANNS,    ''),
    ('COCO 2014', 'Invalid bbox removed',              n_bbox_removed_coco, 'width/height=0 or out-of-bounds'),
    ('COCO 2014', 'Duplicate ann_ids removed',         n_coco_dups,       ''),
    ('COCO 2014', 'Final annotation rows',             len(coco_df),      ''),
    ('COCO 2014', 'Unique images (final)',             coco_df['image_id'].nunique(), ''),
    ('COCO 2014', 'Unique categories (final)',         coco_df['category_name'].nunique(), ''),
    # RefCOCO
    ('RefCOCO',   'Original refs (UNC)',               ORIG_REFCOCO_REFS_UNC,  ''),
    ('RefCOCO',   'Original refs (Google)',             ORIG_REFCOCO_REFS_GGL,  ''),
    ('RefCOCO',   'Original expressions (UNC+Google)',  ORIG_REFCOCO_EXPRS,     'after sentence expansion'),
    ('RefCOCO',   'Removed — missing img/ann in COCO', n_missing_img + n_missing_ann, 'during enrichment join'),
    ('RefCOCO',   'Removed — empty expressions',       n_empty,                ''),
    ('RefCOCO',   'Removed — invalid bboxes',          n_bbox_removed_rc,      ''),
    ('RefCOCO',   'Removed — reference integrity',     n_ref_removed,          ''),
    ('RefCOCO',   'Removed — exact duplicates',        n_exact_dups,           f'key: {DUP_KEY}'),
    ('RefCOCO',   'Final expression rows',             len(refcoco_df),        ''),
    ('RefCOCO',   'Unique images (final)',             refcoco_df['image_id'].nunique(), ''),
    ('RefCOCO',   'Unique ann targets (final)',        refcoco_df['ann_id'].nunique(), ''),
    ('RefCOCO',   'Unique categories (final)',         refcoco_df['category_name'].nunique(), ''),
]

summary_df = pd.DataFrame(summary_rows, columns=['Dataset', 'Metric', 'Count', 'Notes'])
summary_df['Count'] = summary_df['Count'].apply(lambda x: f'{x:,}' if isinstance(x, int) else x)

print('PREPROCESSING SUMMARY')
print('=' * 90)
print(summary_df.to_string(index=False))


PREPROCESSING SUMMARY
  Dataset                            Metric   Count                                             Notes
COCO 2014                   Original images  82,783                                                  
COCO 2014              Original annotations 604,907                                                  
COCO 2014              Invalid bbox removed       1                   width/height=0 or out-of-bounds
COCO 2014         Duplicate ann_ids removed       0                                                  
COCO 2014             Final annotation rows 604,906                                                  
COCO 2014             Unique images (final)  82,081                                                  
COCO 2014         Unique categories (final)      80                                                  
  RefCOCO               Original refs (UNC)  50,000                                                  
  RefCOCO            Original refs (Google)  50,000         

In [46]:
print('MODULE-2 PREPROCESSING SUMMARY')
print('=' * 90)
print(summary_df.to_string(index=False))


MODULE-2 PREPROCESSING SUMMARY
  Dataset                            Metric   Count                                             Notes
COCO 2014                   Original images  82,783                                                  
COCO 2014              Original annotations 604,907                                                  
COCO 2014              Invalid bbox removed       1                   width/height=0 or out-of-bounds
COCO 2014         Duplicate ann_ids removed       0                                                  
COCO 2014             Final annotation rows 604,906                                                  
COCO 2014             Unique images (final)  82,081                                                  
COCO 2014         Unique categories (final)      80                                                  
  RefCOCO               Original refs (UNC)  50,000                                                  
  RefCOCO            Original refs (Google)  50,000

### 11.1 — File Existence & Size Checks

**Expected:** Both CSV files exist, are non-empty, and match the row counts reported above.


In [47]:
print('OUTPUT FILE CHECKS')
print('=' * 60)

for label, path in [
    ('coco_processed.csv   ', COCO_PROCESSED_CSV),
    ('refcoco_processed.csv', REFCOCO_PROCESSED_CSV),
]:
    exists = path.exists()
    size   = path.stat().st_size / (1024 ** 2) if exists else 0
    status = 'OK' if exists and size > 0 else 'PROBLEM'
    print(f'  [{status}] {label} : exists={exists}  size={size:.2f} MB')

print()

# Read back from disk to confirm round-trip integrity
coco_check    = pd.read_csv(COCO_PROCESSED_CSV,    nrows=3)
refcoco_check = pd.read_csv(REFCOCO_PROCESSED_CSV, nrows=3)

print('coco_processed.csv — first 3 rows (read from disk):')
print(coco_check.to_string(index=False))
print()
print('refcoco_processed.csv — first 3 rows (read from disk):')
print(refcoco_check.to_string(index=False))


OUTPUT FILE CHECKS
  [OK] coco_processed.csv    : exists=True  size=52.17 MB
  [OK] refcoco_processed.csv : exists=True  size=28.92 MB

coco_processed.csv — first 3 rows (read from disk):
 image_id                      image_path  width  height  ann_id   bbox_x   bbox_y  bbox_width  bbox_height  category_id category_name
   480023 COCO_train2014_000000480023.jpg    480     640      86 116.9500 305.8600    285.3000     266.0300           58       hot dog
    50518 COCO_train2014_000000050518.jpg    640     480      89 245.5400 208.1700     40.1400      19.1000           58       hot dog
   142589 COCO_train2014_000000142589.jpg    500     375      93 288.4000  18.0700    211.6000     331.3300           58       hot dog

refcoco_processed.csv — first 3 rows (read from disk):
 image_id                      image_path  ann_id   bbox_x   bbox_y  bbox_width  bbox_height  category_id category_name         referring_expression split source
   581857 COCO_train2014_000000581857.jpg 1719310 103.

### 11.2 — Null Check on Critical Columns

**Expected:** Zero nulls in all columns that downstream notebooks depend on.


In [48]:
CRITICAL_COCO_COLS    = ['image_id', 'image_path', 'width', 'height',
                          'ann_id', 'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height',
                          'category_id', 'category_name']

CRITICAL_REFCOCO_COLS = ['image_id', 'image_path', 'ann_id',
                          'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height',
                          'category_id', 'category_name',
                          'referring_expression', 'split', 'source']

print('NULL CHECK — coco_df:')
coco_nulls = coco_df[CRITICAL_COCO_COLS].isnull().sum()
print(coco_nulls.to_string())
all_coco_ok = (coco_nulls == 0).all()
print(f'  All zero nulls: {all_coco_ok}')

print()
print('NULL CHECK — refcoco_df:')
refcoco_nulls = refcoco_df[CRITICAL_REFCOCO_COLS].isnull().sum()
print(refcoco_nulls.to_string())
all_rc_ok = (refcoco_nulls == 0).all()
print(f'  All zero nulls: {all_rc_ok}')


NULL CHECK — coco_df:
image_id         0
image_path       0
width            0
height           0
ann_id           0
bbox_x           0
bbox_y           0
bbox_width       0
bbox_height      0
category_id      0
category_name    0
  All zero nulls: True

NULL CHECK — refcoco_df:
image_id                0
image_path              0
ann_id                  0
bbox_x                  0
bbox_y                  0
bbox_width              0
bbox_height             0
category_id             0
category_name           0
referring_expression    0
split                   0
source                  0
  All zero nulls: True


### 11.3 — Split Distribution in Processed RefCOCO

**Expected:** Train should be the largest split; testA and testB together roughly match val in size.


In [49]:
print('REFCOCO SPLIT x SOURCE DISTRIBUTION (final processed):')
split_dist = (
    refcoco_df.groupby(['split', 'source'])
    .size()
    .unstack(fill_value=0)
)
split_dist['total'] = split_dist.sum(axis=1)
print(split_dist.to_string())

print()
print('REFCOCO CATEGORY DISTRIBUTION (top 20 by expression count):')
cat_dist = (
    refcoco_df['category_name']
    .value_counts()
    .head(20)
    .reset_index()
)
cat_dist.columns = ['category_name', 'expression_count']
print(cat_dist.to_string(index=False))


REFCOCO SPLIT x SOURCE DISTRIBUTION (final processed):
source  google     unc   total
split                         
test     13368       0   13368
testA        0    5349    5349
testB        0    4794    4794
train   107002  113373  220375
val      13414   10268   23682

REFCOCO CATEGORY DISTRIBUTION (top 20 by expression count):
category_name  expression_count
       person            134252
        zebra              5482
     elephant              5480
         bowl              5434
        chair              5412
      giraffe              5306
        pizza              4460
     sandwich              4298
        donut              4136
        couch              4012
          cow              3960
        horse              3844
   motorcycle              3842
        sheep              3818
   teddy bear              3792
          car              3314
     suitcase              3198
          bus              3052
       banana              3032
         cake              

### 11.4 — Sample Processed Rows

**Expected:** Rows show valid image paths, non-zero bboxes, and clean expressions.


In [50]:
print('SAMPLE ROWS — coco_df (5 random):')
print(coco_df.sample(5, random_state=7).to_string(index=False))
print()
print('SAMPLE ROWS — refcoco_df (5 random, spread across splits):')
sample_parts = []
for sp in sorted(refcoco_df['split'].unique()):
    grp = refcoco_df[refcoco_df['split'] == sp]
    sample_parts.append(grp.sample(min(2, len(grp)), random_state=7))
samples = pd.concat(sample_parts).head(5).reset_index(drop=True)
print(samples[['image_id', 'ann_id', 'category_name',
               'referring_expression', 'split', 'source']].to_string(index=False))


SAMPLE ROWS — coco_df (5 random):
 image_id                      image_path  width  height  ann_id   bbox_x   bbox_y  bbox_width  bbox_height  category_id category_name
   352185 COCO_train2014_000000352185.jpg    640     480  411811  94.5500  58.4300     12.0400      22.0000           10 traffic light
   432268 COCO_train2014_000000432268.jpg    640     426  548534 464.6400  35.6900     19.3800      50.0000            1        person
   374102 COCO_train2014_000000374102.jpg    500     333   97965 167.5700 204.8100    148.9600      93.8200           63         couch
   539671 COCO_train2014_000000539671.jpg    640     427  594607   0.0000 217.6600    240.8500     187.1100           25       giraffe
   572439 COCO_train2014_000000572439.jpg    640     478  455662 209.6500  62.5900     93.9400     233.2000            1        person

SAMPLE ROWS — refcoco_df (5 random, spread across splits):
 image_id  ann_id category_name                                   referring_expression split sou